# NB Warm-up 01 — From a CSV link to useful answers

**STAT3009 · Recommender Systems**  
**Suggested time:** 15–20 minutes  
**Version:** Student version

> **Recall → Run → Reflect.** We will revisit last lecture by loading a CSV directly from the web and turning it into a small, analysis-ready pandas table.

By the end of this warm-up, you should be able to:

1. read a CSV URL with `pd.read_csv`;
2. inspect the shape, columns, data types, and first rows of a `DataFrame`;
3. select columns, filter rows, sort results, and reset the index;
4. summarize ratings with `value_counts`, `agg`, and `groupby`.

## The workflow

```text
CSV link  →  pd.read_csv(...)  →  DataFrame  →  inspect  →  select/filter  →  summarize
```

Today we use the public **STAT3009 Netflix course subset**. Each observed rating is a triple

$$
(u, i, r) = (\text{user}, \text{movie}, \text{rating}).
$$

## 1. Import pandas

In [ ]:
import pandas as pd

print("pandas version:", pd.__version__)

## 2. Read a CSV directly from a URL

`pd.read_csv` accepts a local path **or** a web URL. The URL below points to the raw CSV file, so pandas can download and parse it in one step.

We intentionally load only the three columns used in this lecture.

In [ ]:
TRAIN_URL = (
    "https://raw.githubusercontent.com/"
    "statmlben/CUHK-STAT3009/main/dataset/netflix/train.csv"
)

COLUMNS = ["user_id", "movie_id", "rating"]

train = pd.read_csv(TRAIN_URL, usecols=COLUMNS)

### Read the line of code from the inside out

- `TRAIN_URL`: where the file lives;
- `usecols=COLUMNS`: which columns to keep;
- `pd.read_csv(...)`: download and parse the CSV;
- `train = ...`: store the resulting `DataFrame` in a variable.

> If this cell fails, first check the internet connection and then open `TRAIN_URL` in a browser.

## 3. Always inspect before analysing

Start with four questions: **How many rows? Which columns? Which data types? What do the rows look like?**

In [ ]:
print("shape:", train.shape)
print("columns:", train.columns.tolist())
print("data types:")
display(train.dtypes)

In [ ]:
train.head()

The first row represents the triple $(u, i, r)=(1960, 670, 4)$: **user 1960 gave movie 670 a rating of 4**.

> `user_id` and `movie_id` are identifiers, not numerical measurements. For example, movie 100 is not “twice” movie 50.

## 4. Select columns

Single brackets select one column as a `Series`; a list inside the brackets selects a `DataFrame`.

In [ ]:
ratings = train["rating"]
triples = train[["user_id", "movie_id", "rating"]]

print(type(ratings))
print(type(triples))

## 5. Filter rows with a Boolean condition

First create a `True`/`False` mask, then use `.loc[rows, columns]`.

In [ ]:
is_five_star = train["rating"] == 5
is_five_star.head()

In [ ]:
five_star = train.loc[
    is_five_star,
    ["user_id", "movie_id", "rating"],
]

five_star.head()

## 6. Summarize one column

`value_counts` is useful for discrete values such as one-to-five-star ratings.

In [ ]:
rating_counts = train["rating"].value_counts().sort_index()
rating_counts

In [ ]:
rating_summary = train["rating"].agg(["count", "mean", "std", "min", "max"])
rating_summary

## 7. Summarize by user

`groupby` follows the **split → apply → combine** idea:

1. split the rows by `user_id`;
2. calculate the number and mean of ratings for each user;
3. combine the answers into one table.

This table will connect directly to the **user-average baseline** later in the lecture.

In [ ]:
user_summary = (
    train.groupby("user_id", as_index=False)
    .agg(
        n_ratings=("rating", "size"),
        mean_rating=("rating", "mean"),
    )
    .sort_values("n_ratings", ascending=False)
    .reset_index(drop=True)
)

user_summary.head()

## Your turn — a 4-minute challenge

Load the course `test.csv` from `TEST_URL`, then:

1. keep only `user_id`, `movie_id`, and `rating`;
2. inspect its shape and first five rows;
3. create `test_positive` containing only ratings of 4 or 5;
4. calculate the proportion of test ratings that are 4 or 5.

Use the patterns above to write your code in the cell below.

In [ ]:
TEST_URL = (
    "https://raw.githubusercontent.com/"
    "statmlben/CUHK-STAT3009/main/dataset/netflix/test.csv"
)

# Your code here

## Exit ticket

Explain this expression in one sentence:

```python
train.loc[train["rating"] == 5, ["user_id", "movie_id"]]
```

A good answer identifies both **which rows** and **which columns** are returned.

## Compact pandas reference

| Goal | Pattern |
|---|---|
| Read a web CSV | `pd.read_csv(url)` |
| Keep selected columns while reading | `pd.read_csv(url, usecols=cols)` |
| Inspect first rows | `df.head()` |
| Check dimensions | `df.shape` |
| Select one column | `df["rating"]` |
| Select several columns | `df[["user_id", "rating"]]` |
| Filter rows and select columns | `df.loc[condition, columns]` |
| Sort rows | `df.sort_values(...)` |
| Reset the row index | `df.reset_index(drop=True)` |
| Count discrete values | `s.value_counts()` |
| Summarize by group | `df.groupby(...).agg(...)` |

**Next:** use these tables to construct global-, user-, and item-average rating predictions.